In [1]:
%run initialize_environment.py

Environment initializing completed successfully.


## Simple Multi-Agent Design Pattern

### Prerequisite:
Define/Create all the tools required by the system

### 1. Create multiple agents, each with its own list of tools respectively...
Each subagent is then created with `create_agent(...)`:

- `subagent_1` gets access only to `square_root`
- `subagent_2` gets access only to `square`

### 2. Wrap each subagent as a tool
To let another agent use these specialists, each subagent is wrapped inside a new tool (using @tool decorator):

- `call_subagent_1(x)` sends a message to `subagent_1`
- `call_subagent_2(x)` sends a message to `subagent_2`

This means the main agent does not directly perform the math. Instead, it delegates the work to the appropriate subagent.

### 3. Create a main coordinating agent
Finally, `main_agent` is created with:

- the same LLM
- the tools `call_subagent_1` and `call_subagent_2`
- a system prompt telling it when to use them

## Creating subagents

In [2]:
@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [3]:
llm = create_azure_llm()

subagent_1 = create_agent( model=llm, 
    tools=[square_root]
)

subagent_2 = create_agent( model=llm,
    tools=[square]
)

Using chat deployment: lums-gpt-4.1-mini
Endpoint: https://aoai-foundry-swc.openai.azure.com/
API Version: 2025-01-01-preview


In [4]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model=llm,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to " \
                   "calculate the square root or square of a number.")

## Test

In [5]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
pprint(response['messages'][-1].content)

'The square root of 456 is approximately 21.354.'


In [6]:
question = "What is the square of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})
pprint(response['messages'][-1].content)

'The square of 456 is 207936.'
